# Healthcare Cost Analysis

This notebook explores factors that influence individual healthcare costs using the **Medical Cost Personal Dataset**.  
It includes:

- data loading and preprocessing
- exploratory data analysis
- correlation analysis
- linear regression
- residual analysis
- decision tree modeling
- feature importance

**Author:** Isabel Romero


## 1. Import libraries

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score

# display settings
pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")


## 2. Load the dataset

Place `insurance.csv` in the same folder as this notebook, or update the path below.


In [ ]:
df = pd.read_csv("insurance.csv")
df.head()

## 3. Inspect the data

In [ ]:
print("Shape:", df.shape)
print("\nData types:")
print(df.dtypes)
print("\nMissing values:")
print(df.isnull().sum())

In [ ]:
df.describe(include="all")

## 4. Encode categorical variables

For modeling, categorical fields are converted to numeric values:
- `smoker`: yes = 1, no = 0
- `sex`: male = 1, female = 0

A copy of the original dataframe is kept for visualization.


In [ ]:
df_model = df.copy()

df_model["smoker"] = df_model["smoker"].map({"yes": 1, "no": 0})
df_model["sex"] = df_model["sex"].map({"male": 1, "female": 0})

df_model.head()

## 5. Exploratory data analysis

### Average healthcare cost by smoking status

In [ ]:
plt.figure(figsize=(7, 5))
avg_cost = df.groupby("smoker")["charges"].mean().reindex(["no", "yes"])
avg_cost.plot(kind="bar")
plt.title("Average Healthcare Cost by Smoking Status")
plt.xlabel("Smoker")
plt.ylabel("Average Charges")
plt.tight_layout()
plt.show()

### Age vs healthcare cost

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x="age", y="charges", hue="smoker", alpha=0.7)
plt.title("Age vs Healthcare Cost")
plt.xlabel("Age")
plt.ylabel("Charges")
plt.tight_layout()
plt.show()

### BMI vs healthcare cost

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x="bmi", y="charges", hue="smoker", alpha=0.7)
plt.title("BMI vs Healthcare Cost")
plt.xlabel("BMI")
plt.ylabel("Charges")
plt.tight_layout()
plt.show()

### Distribution of healthcare costs

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(df["charges"], bins=30)
plt.title("Distribution of Healthcare Costs")
plt.xlabel("Charges")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

## 6. Correlation heatmap

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(df_model.corr(numeric_only=True), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap of Healthcare Variables")
plt.tight_layout()
plt.show()

## 7. Linear regression model

In [ ]:
X = df_model[["age", "bmi", "children", "smoker", "sex"]]
y = df_model["charges"]

lin_reg = LinearRegression()
lin_reg.fit(X, y)

predictions = lin_reg.predict(X)

rmse = mean_squared_error(y, predictions, squared=False)
r2 = r2_score(y, predictions)

print(f"RMSE: {rmse:,.2f}")
print(f"R^2: {r2:.4f}")

### Regression coefficients

In [ ]:
coef_df = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": lin_reg.coef_
}).sort_values("Coefficient")

coef_df

In [ ]:
plt.figure(figsize=(8, 5))
plt.barh(coef_df["Feature"], coef_df["Coefficient"])
plt.title("Regression Coefficients for Healthcare Cost Prediction")
plt.xlabel("Coefficient Value")
plt.tight_layout()
plt.show()

### Residual plot

In [ ]:
residuals = y - predictions

plt.figure(figsize=(8, 5))
plt.scatter(predictions, residuals, alpha=0.6)
plt.axhline(0, linestyle="--")
plt.title("Residual Plot for Linear Regression Model")
plt.xlabel("Predicted Charges")
plt.ylabel("Residuals")
plt.tight_layout()
plt.show()

## 8. Decision tree model

In [ ]:
tree_model = DecisionTreeRegressor(random_state=42, max_depth=4)
tree_model.fit(X, y)

tree_predictions = tree_model.predict(X)
tree_rmse = mean_squared_error(y, tree_predictions, squared=False)
tree_r2 = r2_score(y, tree_predictions)

print(f"Decision Tree RMSE: {tree_rmse:,.2f}")
print(f"Decision Tree R^2: {tree_r2:.4f}")

### Feature importance

In [ ]:
importance_df = pd.DataFrame({
    "Feature": X.columns,
    "Importance": tree_model.feature_importances_
}).sort_values("Importance")

importance_df

In [ ]:
plt.figure(figsize=(8, 5))
plt.barh(importance_df["Feature"], importance_df["Importance"])
plt.title("Decision Tree Feature Importance")
plt.xlabel("Importance Score")
plt.tight_layout()
plt.show()

## 9. Key findings

- **Smoking status** is the strongest predictor of healthcare costs.
- **Age** and **BMI** also contribute meaningfully to higher charges.
- **Children** has a smaller effect.
- **Sex** appears to have limited predictive value in this simplified model.
- The histogram shows that healthcare charges are **right-skewed**, with a small number of very high-cost observations.

## 10. Notes

- This notebook uses a simple regression and decision tree for interpretability.
- Additional improvements could include train/test split evaluation, cross-validation, and more advanced models such as random forest or gradient boosting.
